# Part 1: Analysis workflow

In this part we are going to analyze our dataset. The task will be divided in two parts:

1. Fill histograms with the muon variables without applying any selection to the muon: this will allow us to understand the distribution of the properties that characterize the muons.
2. Apply a selection to the muons and isolate the muons coming from a Z boson: the objective is to obtain a clean Z boson peak to be able to fit it and extract its properties.

In [ ]:
cd ~/work/CmsOpenData/RDFAnalyzer;

## 1. Import ROOT and necessary tools

In [ ]:
# Import ROOT (ROOT is a analysis package used to handle 
# the tree files, but also histograms, etc.)
import ROOT as R
from utils import histpars, loadRDF, plot, DATA_PATH
%jsroot off

In [ ]:
R.gROOT.ProcessLine('.L ./_tdrstyle.C')
R.setTDRStyle()
R.gROOT.SetBatch(1)
R.gROOT.SetMustClean(False)

## 2. Plot raw data (without any selection)

To load and manipulate the data, we are going to use the ROOT RDataFrame structure [2]. ROOT's RDataFrame offers a modern, high-level interface for analysis of data stored in TTree , CSV and other data formats, in C++ or Python.

[2] https://root.cern/doc/v628/classROOT_1_1RDataFrame.html

In [ ]:
# Load the data into an RDataFrame structure
df = loadRDF("Events", DATA_PATH)

In this first stage, we want to look at the distribution of some key variables of the muons. The goal is to get familiar with the muon properties and to design the selection that we will apply later.

Below you have a list of the histograms that we are going to produce.

In [ ]:
print("\nList of histograms to be produced:\n")
for var in histpars:
    print(f"{var:18} : {histpars[var]['name']:40}")

The produced histograms will be saved in a .root file in the temp folder, so that we can plot them later without having to recompute them.

In [ ]:
# Make all histograms
# -- THIS CELL IS EXPECTED TO TAKE LONGER TO RUN --

hists = {}
for variable in histpars:
    _xlabel = histpars[variable]["xlabel"]
    _bins   = histpars[variable]["bins"]
    _xmin   = histpars[variable]["xmin"]
    _xmax   = histpars[variable]["xmax"]
    h = df.Histo1D(R.RDF.TH1DModel(f"h_{variable}",f";{_xlabel};Events", _bins, _xmin, _xmax), variable)
    hists[variable] = h

print("Writing to file...")
fwrite = R.TFile.Open("temp/hists_raw.root","RECREATE")
for variable in histpars:
    hists[variable].Write()
fwrite.Close()

print("Done!")

Now that the histograms are produced, we can plot the variable we want from the list above.

In [ ]:
# Insert variable to read
variable = "DiMuon_invMass"

fread = R.TFile.Open("temp/hists_raw.root","READ")
h = fread.Get(f"h_{variable}")
h.SetDirectory(0)
fread.Close()

R.gROOT.FindObject(f"c_{variable}") and R.gROOT.FindObject(f"c_{variable}").Close() # Avoid memory conflicts and crashes
c = R.TCanvas(f"c_{variable}","")
c = plot(c, h)
c.Draw()

## 3. Signal vs background discrimination

Now we will apply a selection to remove the background and isolate Z to mu mu events. This selection consists in a series of cuts on the muons' variables:

* _isGlobal_   | Require global muons
* _muonID_     | Apply muon ID: The muon reconstruction in CMS is optimized for maximum efficiency. The collection of reconstructed muons therefore contains muons from a large variety of sources, including potentially misreconstructed muons. A muon identification requirement is therefore necessary to select only those muons relevant for a given analysis. A muon selection ID is a set of predefined requirements on the quality properties of the muons (chi2, number of hits...) that removes badly reconstructed muons. We are going to work with 3 different IDs:
    * Loose ID (0)
    * Medium ID (1)
    * Tight ID (2)
* _pt_min_     | Minimum transverse momentum of the muons
* _abseta_max_ | Maximum value for the pseudorrapidity of the muons
* _dz_max_     | Maximum value for the impact parameter in Z axis of the muons
* _dxy_max_    | Maximum value for the impact parameter in XY plane of the muons
* _relIso_max_ | Maximum value of the relative isolation in the tracker (trkIso / pT)

In [ ]:
# Selection parameters
isGlobal   = None
muonId     = 2
pt_min     = 15.
abseta_max = 2.4
dz_max     = 0.1 # suggested: [0.02, 0.2, 0.4]
dxy_max    = 0.1 # suggested: [0.02, 0.1, 0.2]
relIso_max = 0.3 # Try different values

from utils import pogIds
cuts = {
    'GLB' : f'Muon_isGlobal==1' if isGlobal else None,
    'ID'  : f'{pogIds[muonId]}' if muonId>-1 else None,
    'PT'  : f'Muon_pt>{pt_min}',
    'ETA' : f'abs(Muon_eta)<{abseta_max}',
    'DZ'  : f'Muon_dz<{dz_max}',
    'DXY' : f'Muon_dxy<{dxy_max}',
    'ISO' : f'Muon_tkRelIso<{relIso_max}'
}

cutstring = ''
for cut in cuts:
    if cuts[cut]: cutstring += f'({cuts[cut]})&&'
cutstring = cutstring[:-2]
print(cutstring)

In [ ]:
df = loadRDF("Events", DATA_PATH)
df = df.Define("Muon_isGoodMuon",f"isGoodMuon({cutstring})")
df = df.Define("SelDiMuon_invMass","invariantMass(Muon_isGoodMuon, Muon_pt, Muon_eta, Muon_phi)")
df = df.Filter("SelDiMuon_invMass>=0.")
for variable in histpars:
    if variable.startswith("Muon_"):
        df = df.Define(f"Sel{variable}", f"({variable})[Muon_isGoodMuon]")

In [ ]:
# Make all histograms
# -- THIS CELL IS EXPECTED TO TAKE LONGER TO RUN --

hists_sel = {}
for variable in histpars:
    if variable not in df.GetColumnNames():
        print(f"Warning: {variable} not in DataFrame")
        continue
    _xlabel = histpars[variable]["xlabel"]
    _bins   = histpars[variable]["bins"]
    _xmin   = histpars[variable]["xmin"]
    _xmax   = histpars[variable]["xmax"]
    h = df.Histo1D(R.RDF.TH1DModel(f"h_{variable}",f";{_xlabel};Events", _bins, _xmin, _xmax), f"Sel{variable}")
    hists_sel[variable] = h

print("Writing to file...")
fwrite_sel = R.TFile.Open("temp/hists_selection.root","RECREATE")
for variable in histpars:
    hists_sel[variable].Write()
fwrite_sel.Close()

print("Done!")

In [ ]:
# Insert variable to read
variable = "DiMuon_invMass"

fread = R.TFile.Open("temp/hists_raw.root","READ")
h_raw = fread.Get(f"h_{variable}")
h_raw.SetDirectory(0)
fread.Close()

fread = R.TFile.Open("temp/hists_selection.root","READ")
h_sel = fread.Get(f"h_{variable}")
h_sel.SetDirectory(0)
fread.Close()

R.gROOT.FindObject(f"c_{variable}") and R.gROOT.FindObject(f"c_{variable}").Close() # Avoid memory conflicts and crashes
c = R.TCanvas(f"c_{variable}","")
c = plot(c, [h_raw, h_sel], logy=True)
c.Draw()

After an optimized selection, the dimuon invariant mass shoud show a clear peak at around 90 GeV, with almost 0 background.
These histogram will be taken as input for next part.

[Go to Part 2: Measurement of the Z-boson mass and decay width.]((./Part2_Z_boson_fit.ipynb))